# Filtered Search with Milvus and OpenAI
### Finding your next movie

In this notebook we will be going over generating embeddings of movie descriptions with OpenAI and using those embeddings within Milvus to find relevant movies. To narrow our search results and try something new, we are going to be using filtering to do metadata searches. The dataset in this example is sourced from HuggingFace datasets, and contains a little over 8 thousand movie entries.

Lets begin by first downloading the required libraries for this notebook:
- `openai` is used for communicating with the OpenAI embedding service
- `pymilvus` is used for communicating with the Milvus server
- `datasets` is used for downloading the dataset
- `tqdm` is used for the progress bars


In [1]:
! pip install openai pymilvus datasets tqdm

  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyarrow-21.0.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (3.3 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (7.2 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached frozenlist-1.7.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (18 kB)
  Using cached propcache-0.3.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached yarl-1.20.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (73 kB)
Using cached yarl-1.20.1-cp313-cp313-macosx_11_0_arm64.whl (88 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached frozenlist-1.7.0-cp313-cp313-macosx_11_0_arm64.whl (45 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6

With the required packages installed we can get started. Lets begin by launching the Milvus service. The file being run is the `docker-compose.yaml` found in the folder of this file. This command launches a Milvus standalone instance which we will use for this test.  

In [2]:
! docker compose up -d

WARN[0000] /Users/jung/Projects/Python/msds-420/Assignment_7/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
[+] Running 4/4
 ✔ Container milvus-etcd        Running                                    0.0s 
 ✔ Container milvus-minio       Running                                    0.0s 
 ✔ Container milvus-standalone  Running                                    0.0s 
 ✔ Container attu               Running                                    0.0s 


With Milvus running we can setup our global variables:
- HOST: The Milvus host address
- PORT: The Milvus port number
- COLLECTION_NAME: What to name the collection within Milvus
- DIMENSION: The dimension of the embeddings
- OPENAI_ENGINE: Which embedding model to use
- openai.api_key: Your OpenAI account key
- INDEX_PARAM: The index settings to use for the collection
- QUERY_PARAM: The search parameters to use
- BATCH_SIZE: How many movies to embed and insert at once

In [12]:
import openai

HOST = 'localhost'
PORT = 19530
COLLECTION_NAME = 'movie_search'
DIMENSION = 1536
OPENAI_ENGINE = 'text-embedding-3-small'
openai.api_key = 'sk-your_key'

INDEX_PARAM = {
    'metric_type':'L2',
    'index_type':"HNSW",
    'params':{'M': 8, 'efConstruction': 64}
}

QUERY_PARAM = {
    "metric_type": "L2",
    "params": {"ef": 64},
}

BATCH_SIZE = 1000

In [4]:
from pymilvus import connections, utility, FieldSchema, Collection, CollectionSchema, DataType

# Connect to Milvus Database
connections.connect(host=HOST, port=PORT)

/Users/jung/Projects/Python/msds-420/venv/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at schema.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/jung/Projects/Python/msds-420/venv/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/jung/Projects/Python/msds-420/venv/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at milvus.proto. Please update the gencode to avoid compatibility violations in the next

In [5]:
# Remove collection if it already exists
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)

In [6]:
# Create collection which includes the id, title, and embedding.
fields = [
    FieldSchema(name='id', dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name='title', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='type', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='release_year', dtype=DataType.INT64),
    FieldSchema(name='rating', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='description', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='embedding', dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)
]
schema = CollectionSchema(fields=fields)
collection = Collection(name=COLLECTION_NAME, schema=schema)

In [7]:
# Create the index on the collection and load it.
collection.create_index(field_name="embedding", index_params=INDEX_PARAM)
collection.load()

## Dataset
With Milvus up and running we can begin grabbing our data. Hugging Face Datasets is a hub that holds many different user datasets, and for this example we are using HuggingLearners's netflix-shows dataset. This dataset contains movies and their metadata pairs for over 8 thousand movies. We are going to embed each description and store it within Milvus along with its title, type, release_year and rating.

In [8]:
import datasets

# Download the dataset 
dataset = datasets.load_dataset('hugginglearners/netflix-shows', split='train')

README.md: 0.00B [00:00, ?B/s]

netflix_titles.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8807 [00:00<?, ? examples/s]

## Insert the Data
Now that we have our data on our machine we can begin embedding it and inserting it into Milvus. The embedding function takes in text and returns the embeddings in a list format. 

In [9]:
# Simple function that converts the texts to embeddings
def embed(texts):
    embeddings = openai.Embedding.create(
        input=texts,
        engine=OPENAI_ENGINE
    )
    return [x['embedding'] for x in embeddings['data']]


This next step does the actual inserting. We iterate through all the entries and create batches that we insert once we hit our set batch size. After the loop is over we insert the last remaning batch if it exists. 

In [14]:
from tqdm import tqdm

# Initialize data structure (excluding auto-generated id field)
data = [
    [], # title
    [], # type  
    [], # release_year
    [], # rating
    [], # description
    [], # embedding
]

descriptions_batch = []

for i in tqdm(range(0, len(dataset)), desc="Processing dataset"):
    data[0].append(dataset[i]['title'] or '')           # title
    data[1].append(dataset[i]['type'] or '')            # type
    data[2].append(dataset[i]['release_year'] or -1)    # release_year
    data[3].append(dataset[i]['rating'] or '')          # rating
    data[4].append(dataset[i]['description'] or '')     # description
    
    descriptions_batch.append(dataset[i]['description'] or '')
    
    if len(data[0]) % BATCH_SIZE == 0:
        # Get embeddings
        embeddings = embed(descriptions_batch)
        data[5].extend(embeddings)  # Add to embedding field
        
        # Insert (Milvus will auto-generate IDs)
        collection.insert(data)
        print(f"✅ Inserted batch of {len(data[0])} items")
        
        # Reset
        data = [[], [], [], [], [], []]
        descriptions_batch = []

# Handle remainder
if len(data[0]) != 0:
    embeddings = embed(descriptions_batch)
    data[5].extend(embeddings)
    collection.insert(data)
    print(f"✅ Inserted final batch of {len(data[0])} items")


Processing dataset:   1%|          | 100/8807 [00:00<01:06, 130.84it/s]

✅ Inserted batch of 100 items


Processing dataset:   2%|▏         | 200/8807 [00:01<00:53, 160.91it/s]

✅ Inserted batch of 100 items


Processing dataset:   3%|▎         | 300/8807 [00:01<00:53, 158.32it/s]

...

Processing dataset:  99%|█████████▉| 8700/8807 [00:53<00:00, 153.88it/s]

✅ Inserted batch of 100 items


Processing dataset: 100%|██████████| 8807/8807 [00:53<00:00, 163.51it/s]

✅ Inserted batch of 100 items


✅ Inserted final batch of 7 items


## Query the Database
With our data safely inserted in Milvus, we can now perform a query. The query takes in a tuple of the movie description you are searching for an the filter to use. More info about the filter can be found [here](https://milvus.io/docs/boolean.md). The search first prints out your description and filter expression. After that for each result we print the score, title, type, release year, rating, and description of the result movies. 

In [15]:
import textwrap

def query(query, top_k = 5):
    text, expr = query
    res = collection.search(embed(text), anns_field='embedding', expr = expr, param=QUERY_PARAM, limit = top_k, output_fields=['title', 'type', 'release_year', 'rating', 'description'])
    for i, hit in enumerate(res):
        print('Description:', text, 'Expression:', expr)
        print('Results:')
        for ii, hits in enumerate(hit):
            print('\t' + 'Rank:', ii + 1, 'Score:', hits.score, 'Title:', hits.entity.get('title'))
            print('\t\t' + 'Type:', hits.entity.get('type'), 'Release Year:', hits.entity.get('release_year'), 'Rating:', hits.entity.get('rating'))
            print(textwrap.fill(hits.entity.get('description'), 88))
            print()

my_query = ('movie about a fluffly animal', 'release_year < 2019 and rating like \"PG%\"')

query(my_query)

Description: movie about a fluffly animal Expression: release_year < 2019 and rating like "PG%"
Results:
	Rank: 1 Score: 0.30387428402900696 Title: The Lamb
		Type: Movie Release Year: 2017 Rating: PG
A big-dreaming donkey escapes his menial existence and befriends some free-spirited
animal pals in this imaginative retelling of the Nativity Story.

	Rank: 2 Score: 0.33846741914749146 Title: Puss in Boots
		Type: Movie Release Year: 2011 Rating: PG
The fabled feline heads to the Land of Giants with friends Humpty Dumpty and Kitty
Softpaws on a quest to nab its greatest treasure: the Golden Goose.

	Rank: 3 Score: 0.34727710485458374 Title: Open Season 2
		Type: Movie Release Year: 2008 Rating: PG
Elliot the buck and his forest-dwelling cohorts must rescue their dachshund pal from
some spoiled pets bent on returning him to domesticity.

	Rank: 4 Score: 0.3476099669933319 Title: Stuart Little 2
		Type: Movie Release Year: 2002 Rating: PG
Zany misadventures are in store as lovable city mou

In [22]:
my_query = ("boring slow predictable cliché unoriginal forgettable mediocre", "rating in ['PG', 'PG-13']")
query(my_query)

Description: boring slow predictable cliché unoriginal forgettable mediocre Expression: rating in ['PG', 'PG-13']
Results:
	Rank: 1 Score: 0.3994305431842804 Title: Dumb and Dumberer: When Harry Met Lloyd
		Type: Movie Release Year: 2003 Rating: PG-13
This wacky prequel to the 1994 blockbuster goes back to the lame-brained title
characters' days as classmates at a Rhode Island high school.

	Rank: 2 Score: 0.416044682264328 Title: Manglehorn
		Type: Movie Release Year: 2014 Rating: PG-13
A reclusive small-town locksmith who can't stop writing letters to a lost love meets a
kindly bank teller who challenges him to look to the future.

	Rank: 3 Score: 0.41862738132476807 Title: As Good as It Gets
		Type: Movie Release Year: 1997 Rating: PG-13
The structured world of a sour, obsessive-compulsive author crumbles when he's drawn
into the lives of a stressed-out single mom and his gay neighbor.

	Rank: 4 Score: 0.4258517622947693 Title: Kahlil Gibran's The Prophet
		Type: Movie Release Year: